## test

In [17]:
import h3 as h3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
# from keplergl import KeplerGl
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [18]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [19]:
fe_query = '''
    select
        cluster,
        hex_id,
        executiondate
    from 
            datasets.city_cluster_hex
        where
            city = 'Bangalore'
            and 
                resolution = 8
'''

In [7]:
test_df = pd.read_sql(fe_query, connection)

In [8]:
test_df

,cluster,hex_id,executiondate
0,Nelamangala,8860144847fffff,2023-09-03
1,Bidadi,8860144029fffff,2023-09-03
2,Bangalore Airport,886016908bfffff,2023-09-03
3,Bannerghatta,886014535dfffff,2023-09-03
4,Bannerghatta,88618926d5fffff,2023-09-03
...,...,...,...
3225,Attibele,886014cb65fffff,2023-09-03
3226,Attibele,88618936a3fffff,2023-09-03
3227,Attibele,88618936b1fffff,2023-09-03
3228,Attibele,88618936b7fffff,2023-09-03


In [1]:
print('523452')

523452


In [5]:
sample_query2 = """

SELECT 
    customer_id, drop_place_id, pickup_hex8,
    fe_count,     
    sum(fe_count) over(partition by customer_id, drop_place_id, pickup_hex8) as fe_count_sum,
    sum(fe_count) over(partition by customer_id, pickup_hex8) as fe_sum_at_hex8,
    booked_order_count,     
    sum(booked_order_count) over(partition by customer_id, drop_place_id, pickup_hex8) as booked_order_count_sum,
    sum(booked_order_count) over(partition by customer_id, pickup_hex8) as booked_order_at_hex8,
    dropped_order_count,     
    sum(dropped_order_count) over(partition by customer_id, drop_place_id, pickup_hex8) as dropped_order_count_sum,
    sum(dropped_order_count) over(partition by customer_id, pickup_hex8) as dropped_order_at_hex8,
    recency_epoch,
    max(recency_epoch) over(partition by customer_id, drop_place_id, pickup_hex8) as recency_epoch_max
from(
    SELECT 
        customer_id,
        drop_place_id,
        pickup_hex8,
        sum(fe_freq) as fe_count,
        sum(booked_order_freq) as booked_order_count,
        sum(dropped_order_freq) as dropped_order_count,
        max(fe_epoch) as recency_epoch
    FROM 
       -- hive.datasets.customer_location_recommendations
       hive.datasets_internal.customer_location_recommendations_immutable_v1_dec
    WHERE 1=1 
        AND city_id = '572ca7ff116b5db3057bd814'
        AND updated_yyyymmdd >= '20240701'
        AND updated_yyyymmdd <= '20241208'
        AND customer_id in (
                            with se as (
                                select 
                                    *
                                from(
                                    select  
                                        yyyymmdd,
                                        hhmmss,
                                        epoch,
                                        (event_props_user_id || '-' || event_props_ct_session_id) as search_unique_id,
                                        event_props_user_id as customer_id,
                                        event_props_current_city as current_city,
                                        event_props_for,
                                        event_props_from,
                                        event_props_screen,
                                        event_props_rank,
                                        event_props_selection_mode,
                                        event_props_lat as drop_lat,
                                        event_props_lon as drop_lon,
                                        json_extract_scalar(event_props, '$.recos_strategy') as strategy,
                                        event_props_place_id as place_id,
                                        row_number() over(partition by event_props_user_id order by epoch asc) as rn
                                    from clevertap.customer_searchaddress_immutable
                                    where yyyymmdd  >= '20241209' and yyyymmdd <= '20241209'
                                        and event_props_current_city = 'Bangalore'
                                        and event_props_selection_mode in ('recent_history', 'favourites')
                                        and event_props_screen = 'search_screen'
                                        and event_props_for = 'drop'
                                        and json_extract_scalar(event_props, '$.recos_strategy') != ''
                                    )
                                where rn = 1 
                            )

                            ,fe as(
                                select 
                                    fe_unique_id, pickup_lat, pickup_lon, pickup_hex8, pickup_hex9
                                from(
                                    select 
                                        (user_id || '-' || ct_session_id) as fe_unique_id,
                                        pickup_location_latitude as pickup_lat, pickup_location_longitude as pickup_lon,
                                        pickup_location_hex_8 as pickup_hex8, pickup_location_hex_9 as pickup_hex9,
                                        row_number() over(partition by user_id, ct_session_id order by epoch) as row 
                                    from canonical.clevertap_customer_fare_estimate
                                    where yyyymmdd  >= '20241209' and yyyymmdd <= '20241209'
                                    )
                                where row = 1 
                            )

                            select 
                                customer_id
                            from se
                            join fe
                                on se.search_unique_id = fe.fe_unique_id
                            
                            )
    GROUP BY 1,2,3
    ORDER BY 1,2,3
    )
order by 6

"""

In [22]:
view = """

-- select * from reports_internal.rtu_customer_rf_recos_dec10_view
select * from  reports_internal.rtu_customer_rf_recos_dec10_v3
-- rtu_customer_rf_recos_dec10_query2_view


"""

In [23]:

df2 = pd.read_sql(view, connection)
print(df2.shape)
df2.head()


DatabaseError: {'message': "line 4:16: Table 'hive.reports_internal.rtu_customer_rf_recos_dec10_v3' does not exist", 'errorCode': 46, 'errorName': 'TABLE_NOT_FOUND', 'errorType': 'USER_ERROR', 'errorLocation': {'lineNumber': 4, 'columnNumber': 16}, 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': "line 4:16: Table 'hive.reports_internal.rtu_customer_rf_recos_dec10_v3' does not exist", 'suppressed': [], 'stack': ['io.trino.sql.analyzer.SemanticExceptions.semanticException(SemanticExceptions.java:48)', 'io.trino.sql.analyzer.SemanticExceptions.semanticException(SemanticExceptions.java:43)', 'io.trino.sql.analyzer.StatementAnalyzer$Visitor.visitTable(StatementAnalyzer.java:1791)', 'io.trino.sql.analyzer.StatementAnalyzer$Visitor.visitTable(StatementAnalyzer.java:445)', 'io.trino.sql.tree.Table.accept(Table.java:60)', 'io.trino.sql.tree.AstVisitor.process(AstVisitor.java:27)', 'io.trino.sql.analyzer.StatementAnalyzer$Visitor.process(StatementAnalyzer.java:462)', 'io.trino.

In [ ]:

# df2.to_parquet('RF_data.parquet',index=False)
df2.to_parquet('RF_data2.parquet',index=False)